In [40]:
import torch # type:ignore
import torchvision # type:ignore
import torch.nn as nn # type:ignore
import torch.optim as optim # type:ignore
from torch.utils.data import DataLoader # type:ignore
import torchvision.transforms as transforms # type:ignore
from torchvision.datasets import ImageFolder # type:ignore

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [42]:
# transforms.Compose([]) -> It groups multiple image transformation steps into a single pipeline. When an image passes through this pipeline, it gets sequentially modified by each operation inside the list.

# transforms.RandomCrop(32, padding=4) -> It pads the original 32*32 pixel image by 4 pixels an all sides (making 40x40) and then randomly crops a 32x32 section out of it.

# transforms.RandomHorizontalFlip() -> Another data augmentation trick. A dog or a car facing left is still a dog or a car when facing right, which helps the model generalize better to new images. Notice this is not applied to transform_test, because we want our evaluation metrics to remain consistent and deterministic.

# transforms.ToTensor() -> Converts a PIL Image or NumPy array (which has pixel values ranging from 0 to 255 with shape [Height, Width, Channel]) into a PyTorch Tensor.It also scales the pixel values down to a floating-point range of [0.0, 1.0] and rearranges the shape to [Channel, Height, Width], which is what PyTorch expects.

# transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)) -> Normalizes the image tensor by subtracting the mean and dividing by the standard deviation for each channel (Red, Green, Blue) of the cifar-10 dataset. This helps in speeding up convergence during training.

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [43]:
# Automatically download CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
print(f"Training samples: {len(trainset)}")
print(f"Testing samples: {len(testset)}")

Training samples: 50000
Testing samples: 10000


In [44]:
# 1. Create DataLoaders to batch and shuffle the data
trainloader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

When I say that I am doing convolution of X layer wise like conv1, conv2, conv3 what I am simply doing is passing a kernal or filter to extract a type of information from the image at that layer like conv1 will create such a filter that it will capture the edges from the image, conv2 will create such a filter that it will capture the curves from the image and conv3 will pass such a filter that it will capture different sections of image like eyes of cat, tails of cat etc etc.

In [45]:
# # 2. Define a simple Convolutional Neural Network (CNN)
# import torch.nn.functional as F

# class SimpleCNN(nn.Module):
#     def __init__(self):
#         super(SimpleCNN, self).__init__()
#         self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
#         self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
#         self.pool = nn.MaxPool2d(2, 2)
#         self.fc1 = nn.Linear(64 * 8 * 8, 512)
#         self.fc2 = nn.Linear(512, 10)

#     def forward(self, x):
#         x = self.pool(F.relu(self.conv1(x)))
#         x = self.pool(F.relu(self.conv2(x)))
#         x = torch.flatten(x, 1)
#         x = F.relu(self.fc1(x))
#         x = self.fc2(x)
#         return x

# net = SimpleCNN().to(device)
# print(net)

In [46]:
import torch.nn.functional as F

class BetterCNN(nn.Module):
    def __init__(self):
        super(BetterCNN, self).__init__()
        # Block 1: 3 channels in -> 32 channels out
        self.conv1 = nn.Conv2d(3,32,kernel_size=3,padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # Block 2: 32 channels in -> 64 channels out
        self.conv2 = nn.Conv2d(32,64,kernel_size=3,padding=1)
        self.bn2 = nn.BatchNorm2d(64) 

        # Block 3: 64 channels in -> 128 channels out
        self.conv3 = nn.Conv2d(64,128,kernel_size=3,padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # Reusable MaxPool layer (halves the image dimensions)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Dropout layer to prevent overfitting (turns off 50% of neurons)
        self.dropout = nn.Dropout(0.3)

        # Calculation for Linear layer size:
        # Starting image: 32x32
        # After pool 1: 16x16
        # After pool 2: 8x8
        # After pool 3: 4x4
        # Total flattened size = 128 channels * 4 * 4 = 2048
        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        # Pass through Conv -> BatchNorm -> ReLU Activation -> MaxPool
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        
        # Flatten the 3D tensor into a 1D vector
        x = torch.flatten(x, 1)

        # Pass through Dropout -> Linear -> ReLU -> Linear
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
# Instantiate the upgraded network
model = BetterCNN().to(device)
print(model)

BetterCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=2048, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
)


In [47]:
# 3. Define the Loss function and Optimizer
criterion = nn.CrossEntropyLoss()
# optimizer = optim.SGD(model.parameters(), lr=0.009, momentum=0.7)
optimizer = optim.RMSprop(model.parameters(), lr=0.0005, alpha=0.9)
# optimizer = optim.Adam(model.parameters(), lr=0.002)

In [48]:
# 4. Training Loop
epochs = 10
for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 199:    # print every 200 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 200:.3f}')
            running_loss = 0.0

print('Finished Training')

[1,   200] loss: 1.762
[1,   400] loss: 1.462
[1,   600] loss: 1.355
[2,   200] loss: 1.193
[2,   400] loss: 1.141
[2,   600] loss: 1.107
[3,   200] loss: 1.006
[3,   400] loss: 0.997
[3,   600] loss: 0.978
[4,   200] loss: 0.947
[4,   400] loss: 0.912
[4,   600] loss: 0.889
[5,   200] loss: 0.861
[5,   400] loss: 0.865
[5,   600] loss: 0.849
[6,   200] loss: 0.821
[6,   400] loss: 0.814
[6,   600] loss: 0.804
[7,   200] loss: 0.776
[7,   400] loss: 0.773
[7,   600] loss: 0.806
[8,   200] loss: 0.760
[8,   400] loss: 0.750
[8,   600] loss: 0.745
[9,   200] loss: 0.734
[9,   400] loss: 0.725
[9,   600] loss: 0.729
[10,   200] loss: 0.693
[10,   400] loss: 0.712
[10,   600] loss: 0.699
Finished Training


In [49]:
# 5. Evaluation Loop on Test Data
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct / total:.2f} %')

Accuracy of the network on the 10000 test images: 71.46 %
